In [ ]:
import pandas as pd

file_path = r"C:\Users\HP\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3\accepted_2007_to_2018Q4.csv.gz"

lgd_cols = ['loan_amnt', 'total_pymnt', 'recoveries', 'grade', 'loan_status']
df_lgd_raw = pd.read_csv(file_path, compression='gzip', usecols=lgd_cols, low_memory=False)

charged_off = df_lgd_raw[df_lgd_raw['loan_status'].str.contains('Charged Off', na=False)].copy()

charged_off['loss_amount'] = (charged_off['loan_amnt'] - charged_off['total_pymnt'] - 
                                charged_off['recoveries']).clip(lower=0)
charged_off['lgd'] = (charged_off['loss_amount'] / charged_off['loan_amnt']).clip(0, 1)

print("--- LGD trung bình theo Grade (dữ liệu thật) ---")
lgd_by_grade = charged_off.groupby('grade')['lgd'].agg(['mean', 'median', 'count'])
print(lgd_by_grade)

lgd_by_grade[['mean']].to_csv("data/lgd_by_grade.csv")
print("\nĐã lưu data/lgd_by_grade.csv")


import numpy as np

df = pd.read_csv("data/lendingclub_with_cate.csv")
lgd_by_grade = pd.read_csv("data/lgd_by_grade.csv").set_index('grade')['mean'].to_dict()

df['lgd'] = df['grade'].map(lgd_by_grade)
df['potential_loss'] = df['loan_amnt'] * df['lgd']

VERIFICATION_COST = 20

df['expected_value_verify'] = df['CATE'] * df['potential_loss'] - VERIFICATION_COST

print("--- Thống kê Expected Value của việc xác minh ---")
print(df['expected_value_verify'].describe())

n_worth_verifying = (df['expected_value_verify'] > 0).sum()
print(f"\nSố khoản vay ĐÁNG xác minh (expected value dương): {n_worth_verifying:,} "
      f"({n_worth_verifying/len(df):.2%})")

print("\n--- % khoản vay đáng xác minh, theo Grade ---")
worth_by_grade = df.groupby('grade').apply(
    lambda g: (g['expected_value_verify'] > 0).mean(), include_groups=False
)
print(worth_by_grade.sort_values(ascending=False))

current_policy_cost = len(df) * VERIFICATION_COST
current_policy_net = df['expected_value_verify'].sum()

selective_df = df[df['expected_value_verify'] > 0]
selective_cost = len(selective_df) * VERIFICATION_COST
selective_net = selective_df['expected_value_verify'].sum()

print(f"\n=== SO SÁNH CHÍNH SÁCH (trên mẫu {len(df):,} khoản vay) ===")
print(f"Xác minh ĐẠI TRÀ (như hiện tại):")
print(f"  Chi phí: ${current_policy_cost:,.0f}")
print(f"  Net value: ${current_policy_net:,.0f}")
print(f"\nXác minh CÓ CHỌN LỌC (chỉ verify khi expected value > 0):")
print(f"  Chi phí: ${selective_cost:,.0f} (giảm {(1-selective_cost/current_policy_cost):.1%})")
print(f"  Net value: ${selective_net:,.0f}")
print(f"  Cải thiện so với đại trà: ${selective_net - current_policy_net:,.0f}")

df.to_csv("data/lendingclub_final_optimization.csv", index=False)
print("\nĐã lưu data/lendingclub_final_optimization.csv")


df = pd.read_csv("data/lendingclub_final_optimization.csv")
VERIFICATION_COST = 20

results = []
for shrinkage in [1.0, 0.75, 0.5, 0.25, 0.1]:
    df['CATE_adj'] = df['CATE'] * shrinkage
    df['ev_adj'] = df['CATE_adj'] * df['potential_loss'] - VERIFICATION_COST
    
    n_worth = (df['ev_adj'] > 0).sum()
    selective_net = df.loc[df['ev_adj'] > 0, 'ev_adj'].sum()
    
    results.append({
        'Mức tin tưởng CATE': f"{int(shrinkage*100)}%",
        'Số khoản đáng xác minh': n_worth,
        '% tổng': f"{n_worth/len(df):.1%}",
        'Net value chọn lọc ($)': round(selective_net)
    })

results_df = pd.DataFrame(results)
print("--- Sensitivity Analysis: Net Value theo mức độ tin tưởng CATE ---")
print(results_df.to_string(index=False))

conservative = df[df['grade'].isin(['F', 'G'])]
conservative_worth = conservative[conservative['expected_value_verify'] > 0]
print(f"\n--- Kịch bản CỰC THẬN TRỌNG: chỉ xác minh Grade F & G ---")
print(f"Số khoản: {len(conservative_worth):,}")
print(f"Net value: ${conservative_worth['expected_value_verify'].sum():,.0f}")
print(f"Chi phí: ${len(conservative_worth)*VERIFICATION_COST:,.0f}")